In [0]:
# ============================================================
# 07_gold_facts
# ============================================================
# Purpose  : Load fact_inspection and fact_violation into Gold
# Reads    : silver.inspections_unified, silver.violations_unified
#            gold.dim_date, gold.dim_location,
#            gold.dim_restaurant, gold.dim_violation
# Writes   : gold.fact_inspection
#            gold.fact_violation
# MUST run after 06_gold_dimensions
# ============================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, DoubleType, LongType

CATALOG = "final_project_databi_group8"

UNIFIED_INSP  = f"{CATALOG}.silver.inspections_unified"
UNIFIED_VIOLS = f"{CATALOG}.silver.violations_unified"
DIM_DATE       = f"{CATALOG}.gold.dim_date"
DIM_LOCATION   = f"{CATALOG}.gold.dim_location"
DIM_RESTAURANT = f"{CATALOG}.gold.dim_restaurant"
DIM_VIOLATION  = f"{CATALOG}.gold.dim_violation"
FACT_INSP      = f"{CATALOG}.gold.fact_inspection"
FACT_VIOL      = f"{CATALOG}.gold.fact_violation"

print(f"Catalog   : {CATALOG}")
print(f"Facts     : {FACT_INSP}")
print(f"          : {FACT_VIOL}")

In [0]:
for tbl in [DIM_DATE, DIM_LOCATION, DIM_RESTAURANT, DIM_VIOLATION]:
    try:
        spark.table(tbl).limit(1).collect()
        print(f" Found: {tbl}")
    except Exception as e:
        raise Exception(
            f"Dimension not found: {tbl}\n"
            "Run 06_gold_dimensions first.\n"
            f"Error: {e}"
        )
print()
print("All dimensions verified — safe to load facts")

In [0]:
# Cache dimension lookups for surrogate key resolution

dim_date_lookup = (
    spark.table(DIM_DATE)
    .select("date_key", "full_date")
)

dim_loc_lookup = (
    spark.table(DIM_LOCATION)
    .select("location_key", "street_address", "zip_code", "city_source")
)

# Current version only for FK assignment
dim_rest_lookup = (
    spark.table(DIM_RESTAURANT)
    .filter(F.col("is_current") == True)
    .select("restaurant_key", "restaurant_name", "street_address", "city_source")
)

dim_viol_lookup = (
    spark.table(DIM_VIOLATION)
    .select("violation_key", "violation_code", "city_source")
)

print(f"dim_date              : {dim_date_lookup.count():,} rows")
print(f"dim_location          : {dim_loc_lookup.count():,} rows")
print(f"dim_restaurant (curr) : {dim_rest_lookup.count():,} rows")
print(f"dim_violation         : {dim_viol_lookup.count():,} rows")

In [0]:
# ── fact_inspection ───────────────────────────────────────────
# Grain: one row per inspection
# Matches PDF model: inspection_fact_key, restaurant_key,
# date_key, location_key + degenerate dims

insp_src = spark.table(UNIFIED_INSP)

fact_insp_df = (
    insp_src
    .join(dim_date_lookup.alias("d"),
          insp_src["inspection_date"] == F.col("d.full_date"),
          how="left")
    .join(dim_loc_lookup.alias("l"),
          (insp_src["street_address"] == F.col("l.street_address")) &
          (insp_src["zip_code"]       == F.col("l.zip_code")) &
          (insp_src["city_source"]    == F.col("l.city_source")),
          how="left")
    .join(dim_rest_lookup.alias("r"),
          (insp_src["restaurant_name"] == F.col("r.restaurant_name")) &
          (insp_src["street_address"]  == F.col("r.street_address")) &
          (insp_src["city_source"]     == F.col("r.city_source")),
          how="left")
    .select(
        F.col("r.restaurant_key"),
        F.col("d.date_key"),
        F.col("l.location_key"),
        insp_src["inspection_id"],
        insp_src["inspection_type"],
        insp_src["inspection_result"],
        insp_src["inspection_score"].cast(DoubleType()),
        insp_src["risk_category"],
        insp_src["violation_count"].cast(IntegerType()),
        insp_src["city_source"],
        insp_src["_batch_id"],
        insp_src["_silver_ts"],
        F.current_timestamp().alias("dw_loaded_ts")
    )
)

# Add inspection_fact_key (bigint surrogate)
w_fact = Window.orderBy("city_source", "inspection_id")
fact_insp_df = fact_insp_df.withColumn(
    "inspection_fact_key",
    F.row_number().over(w_fact).cast(LongType())
)

fact_insp_df = fact_insp_df.select(
    "inspection_fact_key",
    "restaurant_key", "date_key", "location_key",
    "inspection_id", "inspection_type", "inspection_result",
    "inspection_score", "risk_category", "violation_count",
    "city_source", "_batch_id", "_silver_ts", "dw_loaded_ts"
)

null_date = fact_insp_df.filter(F.col("date_key").isNull()).count()
null_loc  = fact_insp_df.filter(F.col("location_key").isNull()).count()
null_rest = fact_insp_df.filter(F.col("restaurant_key").isNull()).count()

print(f"Total fact_inspection rows : {fact_insp_df.count():,}")
print(f"Null date_key              : {null_date:,}  (should be 0)")
print(f"Null location_key          : {null_loc:,}")
print(f"Null restaurant_key        : {null_rest:,}")
display(fact_insp_df.limit(3))

In [0]:
try:
    spark.table(FACT_INSP).limit(1).collect()
    fact_insp_df.createOrReplaceTempView("fact_insp_src")
    spark.sql(f"""
        MERGE INTO {FACT_INSP} AS target
        USING fact_insp_src AS source
        ON  target.inspection_id = source.inspection_id
        AND target.city_source   = source.city_source
        WHEN MATCHED AND (
            target.violation_count != source.violation_count OR
            COALESCE(CAST(target.inspection_score AS STRING), '') !=
            COALESCE(CAST(source.inspection_score AS STRING), '')
        ) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f" {FACT_INSP} — MERGED")
except Exception:
    (
        fact_insp_df
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FACT_INSP)
    )
    print(f" {FACT_INSP} — CREATED (first run)")

count = spark.table(FACT_INSP).count()
print(f"  Rows: {count:,}  (expected ~267,205)")

In [0]:
# ── fact_violation ────────────────────────────────────────────
# Grain: one row per inspection per violation code
# Matches PDF model: violation_fact_key, restaurant_key,
# date_key, violation_key, location_key + degenerate dims

viol_src = spark.table(UNIFIED_VIOLS)

# Pull date_key, location_key, restaurant_key from fact_inspection
fact_insp_lookup = (
    spark.table(FACT_INSP)
    .select(
        "inspection_id", "city_source",
        "date_key", "location_key", "restaurant_key", "inspection_type"
    )
)

fact_viol_df = (
    viol_src
    .join(dim_viol_lookup.alias("dv"),
          (viol_src["violation_code"] == F.col("dv.violation_code")) &
          (viol_src["city_source"]    == F.col("dv.city_source")),
          how="left")
    .join(fact_insp_lookup.alias("fi"),
          (viol_src["inspection_id"] == F.col("fi.inspection_id")) &
          (viol_src["city_source"]   == F.col("fi.city_source")),
          how="left")
    .select(
        F.col("fi.restaurant_key"),
        F.col("fi.date_key"),
        F.col("dv.violation_key"),
        F.col("fi.location_key"),
        viol_src["inspection_id"],
        F.col("fi.inspection_type"),
        viol_src["violation_points"].cast(DoubleType()),
        viol_src["violation_slot"].cast(IntegerType()),
        viol_src["inspector_comment"],
        viol_src["city_source"],
        viol_src["_batch_id"],
        F.lit(None).cast("timestamp").alias("_silver_ts"),
        F.current_timestamp().alias("dw_loaded_ts")
    )
)

# Add violation_fact_key (bigint surrogate)
w_vfact = Window.orderBy("city_source", "inspection_id", "violation_key")
fact_viol_df = fact_viol_df.withColumn(
    "violation_fact_key",
    F.row_number().over(w_vfact).cast(LongType())
)

fact_viol_df = fact_viol_df.select(
    "violation_fact_key",
    "restaurant_key", "date_key", "violation_key", "location_key",
    "inspection_id", "inspection_type",
    "violation_points", "violation_slot", "inspector_comment",
    "city_source", "_batch_id", "_silver_ts", "dw_loaded_ts"
)

null_viol = fact_viol_df.filter(F.col("violation_key").isNull()).count()
null_date = fact_viol_df.filter(F.col("date_key").isNull()).count()
null_rest = fact_viol_df.filter(F.col("restaurant_key").isNull()).count()

print(f"Total fact_violation rows : {fact_viol_df.count():,}")
print(f"Null violation_key        : {null_viol:,}  (should be 0 or near-0)")
print(f"Null date_key             : {null_date:,}  (should be 0)")
print(f"Null restaurant_key       : {null_rest:,}")
display(fact_viol_df.limit(3))

In [0]:
try:
    spark.table(FACT_VIOL).limit(1).collect()
    fact_viol_df.createOrReplaceTempView("fact_viol_src")
    spark.sql(f"""
        MERGE INTO {FACT_VIOL} AS target
        USING fact_viol_src AS source
        ON  target.inspection_id = source.inspection_id
        AND target.violation_key = source.violation_key
        AND target.city_source   = source.city_source
        WHEN MATCHED AND (
            COALESCE(CAST(target.violation_points AS STRING), '') !=
            COALESCE(CAST(source.violation_points AS STRING), '')
        ) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f" {FACT_VIOL} — MERGED")
except Exception:
    (
        fact_viol_df
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FACT_VIOL)
    )
    print(f" {FACT_VIOL} — CREATED (first run)")

count = spark.table(FACT_VIOL).count()
print(f"  Rows: {count:,}  (expected ~1.4M+)")

In [0]:
print("=== GOLD LAYER SUMMARY ===")
tables = {
    "dim_date":        "6,209",
    "dim_location":    "~41,000",
    "dim_violation":   "~115",
    "dim_restaurant":  "~70,000+",
    "fact_inspection": "~267,205",
    "fact_violation":  "~1.4M+",
}
for tbl, expected in tables.items():
    full = f"{CATALOG}.gold.{tbl}"
    try:
        count = spark.table(full).count()
        print(f"  {tbl:<25} : {count:>10,}  (expected {expected})")
    except Exception as e:
        print(f"  {tbl:<25} : ERROR — {e}")

print()
print("=== fact_inspection by city ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                                                     AS total_inspections,
        ROUND(AVG(inspection_score), 2)                              AS avg_score,
        ROUND(AVG(violation_count),  2)                              AS avg_violations,
        SUM(CASE WHEN inspection_result = 'Pass' THEN 1 ELSE 0 END) AS pass_count,
        SUM(CASE WHEN inspection_result = 'Fail' THEN 1 ELSE 0 END) AS fail_count,
        SUM(CASE WHEN date_key       IS NULL THEN 1 ELSE 0 END)     AS null_date_keys,
        SUM(CASE WHEN location_key   IS NULL THEN 1 ELSE 0 END)     AS null_loc_keys,
        SUM(CASE WHEN restaurant_key IS NULL THEN 1 ELSE 0 END)     AS null_rest_keys
    FROM {FACT_INSP}
    GROUP BY city_source
    ORDER BY city_source
"""))

print()
print("=== fact_violation by city ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                                                      AS total_rows,
        COUNT(DISTINCT violation_key)                                 AS unique_violation_keys,
        SUM(CASE WHEN violation_points IS NULL THEN 1 ELSE 0 END)    AS null_points,
        SUM(CASE WHEN violation_slot   IS NULL THEN 1 ELSE 0 END)    AS null_slots
    FROM {FACT_VIOL}
    GROUP BY city_source
    ORDER BY city_source
"""))

print()
print("=== Star schema end-to-end join test ===")
display(spark.sql(f"""
    SELECT
        dr.restaurant_name,
        dr.facility_type,
        dl.city,
        dl.zip_code,
        dd.year,
        dd.month_name,
        fi.inspection_type,
        fi.inspection_result,
        fi.risk_category,
        fi.inspection_score,
        fi.violation_count
    FROM {FACT_INSP}      fi
    JOIN {DIM_RESTAURANT} dr ON fi.restaurant_key = dr.restaurant_key AND dr.is_current = true
    JOIN {DIM_LOCATION}   dl ON fi.location_key   = dl.location_key
    JOIN {DIM_DATE}       dd ON fi.date_key        = dd.date_key
    LIMIT 10
"""))

In [0]:
# Check what cities are in silver unified
display(spark.sql("""
    SELECT city_source, COUNT(*) AS rows
    FROM final_project_databi_group8.silver.inspections_unified
    GROUP BY city_source
"""))

In [0]:
CATALOG = "final_project_databi_group8"

for tbl in [
    f"{CATALOG}.silver.dallas_clean",
    f"{CATALOG}.silver.dallas_violations",
    f"{CATALOG}.silver.dallas_quarantine",
    f"{CATALOG}.bronze.dallas_raw",
]:
    try:
        count = spark.table(tbl).count()
        print(f" {tbl} — {count:,} rows")
    except Exception as e:
        print(f" {tbl} — NOT FOUND")